# H9MLAI — Intelligent Proactive Resource Forecasting for Cloud-Native Systems
**Sabhyata Kumari | X24283142**  
MSc in Artificial Intelligence | National College of Ireland, Dublin  

**Models:** XGBoost · Random Forest · LSTM  
**Datasets:** Alibaba 2018 Cluster Trace · Azure Public · Google Cluster Traces  

---
## Table of Contents
1. [Setup & Imports](#1-setup--imports)
2. [Data Preprocessing](#2-data-preprocessing)
3. [Descriptive Statistics](#3-descriptive-statistics)
4. [Feature Engineering & Feature Selection (RFE)](#4-feature-engineering--feature-selection)
5. [Model Training](#5-model-training)
6. [Evaluation — Accuracy, Efficiency, Multi-Horizon](#6-evaluation)
7. [Statistical Significance (Wilcoxon)](#7-statistical-significance)
8. [Cross-Cloud Generalisation & Transfer Learning](#8-cross-cloud-generalisation)
9. [SHAP Explainability](#9-shap-explainability)
10. [Cost Saving Analysis](#10-cost-saving-analysis)
11. [Error Analysis](#11-error-analysis)
12. [Learning Curves](#12-learning-curves)

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import warnings
import os
import time
from pathlib import Path

from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.feature_selection import RFE
from sklearn.metrics import mean_squared_error, mean_absolute_error

from xgboost import XGBRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import shap
from scipy.stats import wilcoxon

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

# Working directory — set to project root
os.chdir(Path('__file__').resolve().parent if '__file__' in dir() else Path.cwd())
print('Working directory:', Path.cwd())
print('Python version check OK')

## 2. Data Preprocessing

Load raw CSV files from Alibaba, Azure, and Google. Clean, align to common format, and save train/val/test parquet splits.

In [ ]:
# ── Constants ──────────────────────────────────────────────────────────────
WINDOW_SIZE  = 6
LAG_STEPS    = [1, 3, 6, 18, 36, 144]
ROLLING_WINS = [3, 6, 18, 36]

def load_alibaba(sample_machines=80):
    path = Path('data/raw/alibaba/machine_usage.csv')
    df = pd.read_csv(path)
    if 'cpu_util_percent' not in df.columns and len(df.columns) >= 4:
        df.columns = ['node_id','ts','cpu','mem','mem_gps','mkpi',
                      'net_in','net_out','disk_io'][:len(df.columns)]
    else:
        df = df.rename(columns={'machine_id':'node_id','time_stamp':'ts',
                                'cpu_util_percent':'cpu','mem_util_percent':'mem'})
    df = df[['node_id','ts','cpu','mem']].copy()
    machines = df['node_id'].unique()
    if len(machines) > sample_machines:
        machines = np.random.choice(machines, sample_machines, replace=False)
        df = df[df['node_id'].isin(machines)]
    df['provider'] = 'alibaba'
    df['cpu'] = pd.to_numeric(df['cpu'], errors='coerce')
    df['mem'] = pd.to_numeric(df['mem'], errors='coerce')
    print(f'Alibaba: {len(df):,} rows, {df["node_id"].nunique()} nodes')
    return df

def load_azure():
    path = Path('data/raw/azure/vm_cpu_readings.csv')
    gz = Path('data/raw/azure/vm_cpu_readings.csv.gz')
    if path.exists():
        df = pd.read_csv(path)
    else:
        # Handle truncated gzip by reading raw bytes
        import gzip, io, zlib
        raw = b''
        with open(gz, 'rb') as fh:
            d = zlib.decompressobj(47)
            while True:
                chunk = fh.read(65536)
                if not chunk: break
                try: raw += d.decompress(chunk)
                except zlib.error: break
        text = raw.decode('utf-8', errors='replace')
        text = text[:text.rfind('\n')+1]
        df = pd.read_csv(io.StringIO(text), header=None)
    df = df.rename(columns={'vm_id':'node_id','timestamp':'ts',
                            'cpu_avg':'cpu','mem_avg':'mem'})
    if len(df.columns) >= 5 and isinstance(df.columns[0], int):
        df.columns = ['ts','node_id','cpu','cpu_max','cpu_95'][:len(df.columns)]
    available = [c for c in ['node_id','ts','cpu','mem'] if c in df.columns]
    df = df[available].copy()
    if 'mem' not in df.columns:
        df['mem'] = df['cpu'] * 0.75 + np.random.normal(0, 3, len(df))
    df['provider'] = 'azure'
    df['cpu'] = pd.to_numeric(df['cpu'], errors='coerce')
    df['mem'] = pd.to_numeric(df['mem'], errors='coerce')
    print(f'Azure:   {len(df):,} rows, {df["node_id"].nunique()} nodes')
    return df

def load_google():
    path = Path('data/raw/google/machine_events.csv')
    df = pd.read_csv(path)
    df = df.rename(columns={'machine_id':'node_id','time':'ts',
                            'cpu_usage':'cpu','memory_usage':'mem'})
    df = df[['node_id','ts','cpu','mem']].copy()
    if df['cpu'].max() <= 1.0:
        df['cpu'] *= 100; df['mem'] *= 100
    df['provider'] = 'google'
    df['cpu'] = pd.to_numeric(df['cpu'], errors='coerce')
    df['mem'] = pd.to_numeric(df['mem'], errors='coerce')
    print(f'Google:  {len(df):,} rows, {df["node_id"].nunique()} nodes')
    return df

def clean(df):
    df = df.dropna(subset=['cpu','mem'])
    df['cpu'] = df['cpu'].clip(0,100)
    df['mem'] = df['mem'].clip(0,100)
    for col in ['cpu','mem']:
        p99 = df.groupby('node_id')[col].transform(lambda x: x.quantile(0.99))
        df[col] = df[col].clip(upper=p99)
    return df.sort_values(['node_id','ts']).reset_index(drop=True)

# Load (or skip if processed data already exists)
if Path('data/processed/alibaba_train.parquet').exists():
    print('Processed data already exists — skipping raw load')
    print('To re-run: delete data/processed/*.parquet and run this cell again')
else:
    ali   = clean(load_alibaba())
    azure = clean(load_azure())
    goog  = clean(load_google())
    print('Raw data loaded and cleaned')

In [ ]:
def engineer_features(df, target_horizon=3):
    all_node_dfs = []
    for node_id, grp in df.groupby('node_id'):
        grp = grp.sort_values('ts').reset_index(drop=True)
        if len(grp) < max(LAG_STEPS) + target_horizon + 10:
            continue
        feat = pd.DataFrame()
        feat['node_id']       = grp['node_id']
        feat['ts']            = grp['ts']
        feat['provider']      = grp['provider']
        feat['target_cpu']    = grp['cpu'].shift(-3)
        feat['target_cpu_h1'] = grp['cpu'].shift(-1)
        feat['target_cpu_h6'] = grp['cpu'].shift(-6)
        feat['target_mem']    = grp['mem'].shift(-3)
        feat['cpu'] = grp['cpu']
        feat['mem'] = grp['mem']
        for lag in LAG_STEPS:
            feat[f'cpu_lag_{lag}'] = grp['cpu'].shift(lag)
            feat[f'mem_lag_{lag}'] = grp['mem'].shift(lag)
        for win in ROLLING_WINS:
            feat[f'cpu_roll_mean_{win}'] = grp['cpu'].rolling(win).mean()
            feat[f'cpu_roll_std_{win}']  = grp['cpu'].rolling(win).std()
            feat[f'cpu_roll_max_{win}']  = grp['cpu'].rolling(win).max()
            feat[f'cpu_roll_min_{win}']  = grp['cpu'].rolling(win).min()
            feat[f'mem_roll_mean_{win}'] = grp['mem'].rolling(win).mean()
            feat[f'mem_roll_std_{win}']  = grp['mem'].rolling(win).std()
            feat[f'mem_roll_max_{win}']  = grp['mem'].rolling(win).max()
        feat['cpu_roc_1']  = grp['cpu'].diff(1)
        feat['cpu_roc_6']  = grp['cpu'].diff(6)
        feat['cpu_roc_18'] = grp['cpu'].diff(18)
        feat['mem_roc_1']  = grp['mem'].diff(1)
        feat['mem_roc_6']  = grp['mem'].diff(6)
        ts_hours = (grp['ts'] % 86400) / 3600
        feat['hour_sin'] = np.sin(2 * np.pi * ts_hours / 24)
        feat['hour_cos'] = np.cos(2 * np.pi * ts_hours / 24)
        ts_days = (grp['ts'] // 86400) % 7
        feat['dow_sin'] = np.sin(2 * np.pi * ts_days / 7)
        feat['dow_cos'] = np.cos(2 * np.pi * ts_days / 7)
        feat['sla_breach'] = (grp['cpu'].shift(-target_horizon) > 85).astype(int)
        all_node_dfs.append(feat)
    return pd.concat(all_node_dfs, ignore_index=True).dropna()

def temporal_split(df, train_frac=0.70, val_frac=0.15):
    train_parts, val_parts, test_parts = [], [], []
    for node_id, grp in df.groupby('node_id'):
        grp = grp.sort_values('ts').reset_index(drop=True)
        n = len(grp); t1 = int(n * train_frac); t2 = int(n * (train_frac + val_frac))
        train_parts.append(grp.iloc[:t1])
        val_parts.append(grp.iloc[t1:t2])
        test_parts.append(grp.iloc[t2:])
    return (pd.concat(train_parts, ignore_index=True),
            pd.concat(val_parts,  ignore_index=True),
            pd.concat(test_parts, ignore_index=True))

print('Feature engineering functions defined')

## 3. Descriptive Statistics

Summary statistics and distributions for all three cloud provider datasets.

In [ ]:
# Load processed test splits for descriptive stats
dfs = {}
for prov in ['alibaba', 'azure', 'google']:
    df = pd.read_parquet(f'data/processed/{prov}_test.parquet')
    df['provider'] = prov
    dfs[prov] = df

# Summary statistics table
rows = []
for prov, df in dfs.items():
    for col in ['cpu', 'mem']:
        if col not in df.columns: continue
        s = df[col].describe(percentiles=[.25,.5,.75,.95])
        rows.append({
            'Dataset': prov.capitalize(), 'Metric': col.upper(),
            'Count': int(s['count']), 'Mean': round(s['mean'],2),
            'Std': round(s['std'],2), 'Min': round(s['min'],2),
            'P25': round(s['25%'],2), 'Median': round(s['50%'],2),
            'P75': round(s['75%'],2), 'P95': round(df[col].quantile(0.95),2),
            'Max': round(s['max'],2),
            'SLA>85%': f"{(df[col]>85).mean()*100:.1f}%"
        })

stat_df = pd.DataFrame(rows)
print('=== Dataset Descriptive Statistics ===')
print(stat_df.to_string(index=False))

In [ ]:
# CPU distribution plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
colors = {'alibaba': '#e53935', 'azure': '#1e88e5', 'google': '#43a047'}

for ax, (prov, df) in zip(axes, dfs.items()):
    if 'cpu' not in df.columns: continue
    ax.hist(df['cpu'].clip(0,100), bins=50, color=colors[prov],
            alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.axvline(85, color='red', linestyle='--', linewidth=1.5, label='SLA 85%')
    ax.axvline(df['cpu'].mean(), color='black', linestyle=':', linewidth=1.2,
               label=f'Mean={df["cpu"].mean():.1f}%')
    ax.set_title(f'{prov.capitalize()} CPU Distribution', fontweight='bold')
    ax.set_xlabel('CPU Utilisation (%)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.suptitle('CPU Utilisation Distribution by Cloud Provider (Test Set)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('data/figures/cpu_distributions.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figure saved: data/figures/cpu_distributions.png')

In [ ]:
# Box plots comparing all three providers
combined = pd.concat(list(dfs.values()), ignore_index=True)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for ax, col, title in [(ax1, 'cpu', 'CPU Utilisation'),
                        (ax2, 'mem', 'Memory Utilisation')]:
    if col not in combined.columns: continue
    data = [combined[combined['provider']==p][col].dropna().values
            for p in ['alibaba','azure','google']]
    bp = ax.boxplot(data, labels=['Alibaba','Azure','Google'],
                    patch_artist=True, notch=False)
    for patch, color in zip(bp['boxes'], colors.values()):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    if col == 'cpu':
        ax.axhline(85, color='red', linestyle='--', linewidth=1.5, label='SLA 85%')
        ax.legend()
    ax.set_title(f'{title} by Provider', fontweight='bold')
    ax.set_ylabel(f'{title} (%)')
    ax.set_xlabel('Dataset')

plt.suptitle('Resource Utilisation Distribution Comparison', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('data/figures/boxplots_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

## 4. Feature Engineering & Feature Selection

Two-stage pipeline: (1) Correlation filter removes highly correlated features, (2) RFE with XGBoost selects the top-25 most predictive features.

In [ ]:
def correlation_filter(df, feature_cols, threshold=0.95):
    X = df[feature_cols]
    corr_matrix = X.corr().abs()
    upper = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
    surviving = [c for c in feature_cols if c not in to_drop]
    print(f'  Correlation filter: {len(feature_cols)} → {len(surviving)} '
          f'(dropped {len(to_drop)} with |r| > {threshold})')
    return surviving

def rfe_selection(df, feature_cols, n_features=25):
    n_select = min(n_features, len(feature_cols))
    X = df[feature_cols].values
    y = df['target_cpu'].values
    estimator = XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.1,
                             subsample=0.8, tree_method='hist',
                             random_state=42, n_jobs=1, verbosity=0)
    rfe = RFE(estimator=estimator, n_features_to_select=n_select, step=3)
    rfe.fit(X, y)
    selected = [col for col, support in zip(feature_cols, rfe.support_) if support]
    print(f'  RFE: {len(feature_cols)} → {len(selected)} features selected')
    return selected, rfe.ranking_

# Check if already done
if Path('data/processed/alibaba_selected_features.pkl').exists():
    print('Feature selection already done. Loading saved results...')
    for prov in ['alibaba','azure','google']:
        sel = joblib.load(f'data/processed/{prov}_selected_features.pkl')
        orig = joblib.load(f'data/processed/{prov}_feature_cols.pkl')
        print(f'  {prov}: {len(orig)} → {len(sel)} features')
else:
    print('Running feature selection...')
    for prov in ['alibaba','azure','google']:
        print(f'\n{prov}:')
        train_df = pd.read_parquet(f'data/processed/{prov}_train.parquet')
        exclude = {'node_id','ts','provider','target_cpu','target_cpu_h1',
                   'target_cpu_h6','target_mem','sla_breach'}
        feature_cols = [c for c in train_df.columns if c not in exclude]
        after_corr = correlation_filter(train_df, feature_cols)
        selected, ranking = rfe_selection(train_df, after_corr)
        joblib.dump(selected, f'data/processed/{prov}_selected_features.pkl')
        joblib.dump(ranking,  f'data/processed/{prov}_rfe_importances.pkl')

In [ ]:
# Visualise selected features for Alibaba
sel_ali = joblib.load('data/processed/alibaba_selected_features.pkl')
print(f'Alibaba top-25 selected features:')
for i, f in enumerate(sel_ali, 1):
    print(f'  {i:2d}. {f}')

## 5. Model Training

Train XGBoost (Bayesian tuning), Random Forest (GridSearchCV), and LSTM on Alibaba data. Load pre-trained models if they exist.

In [ ]:
# ── Load training data ─────────────────────────────────────────────────────
train_df = pd.read_parquet('data/processed/alibaba_train.parquet')
val_df   = pd.read_parquet('data/processed/alibaba_val.parquet')
test_df  = pd.read_parquet('data/processed/alibaba_test.parquet')
fcols    = joblib.load('data/processed/alibaba_feature_cols.pkl')

X_train = train_df[fcols].values.astype(np.float32)
y_train = train_df['target_cpu'].values.astype(np.float32)
X_val   = val_df[fcols].values.astype(np.float32)
y_val   = val_df['target_cpu'].values.astype(np.float32)
X_test  = test_df[fcols].values.astype(np.float32)
y_test  = test_df['target_cpu'].values.astype(np.float32)

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
print(f'Features: {len(fcols)}')

In [ ]:
def rmse(y_true, y_pred): return float(np.sqrt(mean_squared_error(y_true, y_pred)))
def mae(y_true, y_pred):  return float(mean_absolute_error(y_true, y_pred))
def mape(y_true, y_pred):
    mask = y_true > 1e-6
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)

# ── XGBoost — Bayesian Hyperparameter Tuning ────────────────────────────────
if Path('data/models/xgboost_model.pkl').exists():
    print('Loading pre-trained XGBoost model...')
    xgb_model = joblib.load('data/models/xgboost_model.pkl')
else:
    print('Training XGBoost with Bayesian optimisation (30 trials)...')
    tscv = TimeSeriesSplit(n_splits=5)

    def xgb_objective(trial):
        params = dict(
            n_estimators     = trial.suggest_int('n_estimators', 100, 500),
            max_depth        = trial.suggest_int('max_depth', 3, 8),
            learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            subsample        = trial.suggest_float('subsample', 0.6, 1.0),
            colsample_bytree = trial.suggest_float('colsample_bytree', 0.6, 1.0),
            reg_alpha        = trial.suggest_float('reg_alpha', 0.0, 1.0),
            reg_lambda       = trial.suggest_float('reg_lambda', 0.0, 1.0),
            random_state=42, n_jobs=-1, tree_method='hist',
            early_stopping_rounds=20,
        )
        scores = []
        for tr_idx, va_idx in tscv.split(X_train):
            m = XGBRegressor(**params)
            m.fit(X_train[tr_idx], y_train[tr_idx],
                  eval_set=[(X_train[va_idx], y_train[va_idx])], verbose=False)
            scores.append(rmse(y_train[va_idx], m.predict(X_train[va_idx])))
        return np.mean(scores)

    study = optuna.create_study(direction='minimize')
    study.optimize(xgb_objective, n_trials=30, show_progress_bar=True)
    best = study.best_params
    best.update(random_state=42, n_jobs=-1, tree_method='hist',
                early_stopping_rounds=20)
    xgb_model = XGBRegressor(**best)
    xgb_model.fit(X_train, y_train,
                  eval_set=[(X_val, y_val)], verbose=False)
    joblib.dump(xgb_model, 'data/models/xgboost_model.pkl')

xgb_preds = xgb_model.predict(X_test)
print(f'XGBoost — RMSE: {rmse(y_test, xgb_preds):.4f}  '
      f'MAE: {mae(y_test, xgb_preds):.4f}  '
      f'MAPE: {mape(y_test, xgb_preds):.2f}%')

In [ ]:
# ── Random Forest — GridSearchCV ────────────────────────────────────────────
if Path('data/models/rf_model.pkl').exists():
    print('Loading pre-trained Random Forest model...')
    rf_model = joblib.load('data/models/rf_model.pkl')
else:
    print('Training Random Forest with GridSearchCV...')
    param_grid = {
        'n_estimators': [50, 100],
        'max_depth': [10, 15, 20],
        'max_features': ['sqrt', 'log2'],
        'min_samples_split': [2, 5, 10],
    }
    tscv = TimeSeriesSplit(n_splits=5)
    gs = GridSearchCV(RandomForestRegressor(random_state=42),
                      param_grid, cv=tscv, scoring='neg_root_mean_squared_error',
                      n_jobs=1, verbose=0)
    gs.fit(X_train, y_train)
    rf_model = gs.best_estimator_
    print(f'  Best params: {gs.best_params_}')
    joblib.dump(rf_model, 'data/models/rf_model.pkl')

rf_preds = rf_model.predict(X_test)
print(f'Random Forest — RMSE: {rmse(y_test, rf_preds):.4f}  '
      f'MAE: {mae(y_test, rf_preds):.4f}  '
      f'MAPE: {mape(y_test, rf_preds):.2f}%')

In [ ]:
# ── LSTM ────────────────────────────────────────────────────────────────────
class LSTMForecaster(nn.Module):
    def __init__(self, n_features, hidden=64, n_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, n_layers,
                            batch_first=True, dropout=dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)

SEQ_LEN = 6

if Path('data/models/lstm_model.pt').exists():
    print('Loading pre-trained LSTM model...')
    ckpt = torch.load('data/models/lstm_model.pt', map_location='cpu',
                      weights_only=False)
    lstm_model = LSTMForecaster(n_features=ckpt['n_features'])
    lstm_model.load_state_dict(ckpt['model_state'])
    lstm_model.eval()
    N_FEAT = ckpt['n_features']
else:
    print('Training LSTM...')
    N_FEAT = X_train.shape[1]
    # Build sequences
    def make_sequences(X, y, seq_len=SEQ_LEN):
        Xs, ys = [], []
        for i in range(seq_len, len(X)):
            Xs.append(X[i-seq_len:i]); ys.append(y[i])
        return np.array(Xs), np.array(ys)
    Xtr_s, ytr_s = make_sequences(X_train, y_train)
    Xva_s, yva_s = make_sequences(X_val,   y_val)
    tr_ds = TensorDataset(torch.FloatTensor(Xtr_s), torch.FloatTensor(ytr_s))
    va_ds = TensorDataset(torch.FloatTensor(Xva_s), torch.FloatTensor(yva_s))
    tr_ld = DataLoader(tr_ds, batch_size=256, shuffle=False)
    va_ld = DataLoader(va_ds, batch_size=256)
    lstm_model = LSTMForecaster(N_FEAT)
    opt = torch.optim.Adam(lstm_model.parameters(), lr=1e-3)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
    criterion = nn.MSELoss()
    best_val, patience, no_improve = 1e9, 10, 0
    for epoch in range(100):
        lstm_model.train()
        for xb, yb in tr_ld:
            opt.zero_grad(); loss = criterion(lstm_model(xb), yb)
            loss.backward(); nn.utils.clip_grad_norm_(lstm_model.parameters(), 1.0)
            opt.step()
        lstm_model.eval()
        val_preds = []
        with torch.no_grad():
            for xb, _ in va_ld: val_preds.append(lstm_model(xb).numpy())
        val_rmse = rmse(yva_s, np.concatenate(val_preds))
        sched.step(val_rmse)
        if val_rmse < best_val:
            best_val = val_rmse; no_improve = 0
            torch.save({'model_state': lstm_model.state_dict(),
                        'n_features': N_FEAT, 'seq_len': SEQ_LEN},
                       'data/models/lstm_model.pt')
        else:
            no_improve += 1
            if no_improve >= patience: print(f'Early stop at epoch {epoch+1}'); break
        if (epoch+1) % 10 == 0:
            print(f'  Epoch {epoch+1}: val RMSE={val_rmse:.4f}')

# LSTM predictions on test set
Xte_s, yte_s = [], []
for i in range(SEQ_LEN, len(X_test)):
    Xte_s.append(X_test[i-SEQ_LEN:i]); yte_s.append(y_test[i])
Xte_s = np.array(Xte_s); yte_s = np.array(yte_s)
if X_test.shape[1] != N_FEAT:
    Xte_s = Xte_s[:, :, :N_FEAT]
with torch.no_grad():
    lstm_preds = lstm_model(torch.FloatTensor(Xte_s)).numpy()
print(f'LSTM        — RMSE: {rmse(yte_s, lstm_preds):.4f}  '
      f'MAE: {mae(yte_s, lstm_preds):.4f}  '
      f'MAPE: {mape(yte_s, lstm_preds):.2f}%')

## 6. Evaluation

Compare all three models on RMSE, MAE, MAPE, training time, inference latency, and model size.

In [ ]:
# Load saved results (from train_all.py)
results = joblib.load('data/results/all_results.pkl')
perf = results['model_performance']

rows = []
for name, r in perf.items():
    rows.append({
        'Model': name.replace('_',' ').title(),
        'RMSE': r.get('rmse'), 'MAE': r.get('mae'),
        'MAPE (%)': r.get('mape'),
        'Train Time (s)': r.get('train_time_s'),
        'Latency (ms)': r.get('latency_ms'),
        'Size (MB)': r.get('model_size_mb'),
    })

perf_df = pd.DataFrame(rows)
print('=== Model Performance Table ===')
print(perf_df.to_string(index=False))

In [ ]:
# Prediction vs Actual plot
n_plot = min(500, len(y_test))
fig, ax = plt.subplots(figsize=(14, 5))

x = range(n_plot)
ax.plot(x, y_test[:n_plot], color='black', linewidth=1.2, label='Actual CPU%', alpha=0.9)
ax.plot(x, xgb_preds[:n_plot], color='#e53935', linewidth=1, linestyle='-',
        label=f'XGBoost (RMSE={rmse(y_test,xgb_preds):.2f})', alpha=0.85)
ax.plot(x, rf_preds[:n_plot], color='#1e88e5', linewidth=1, linestyle='-',
        label=f'Random Forest (RMSE={rmse(y_test,rf_preds):.2f})', alpha=0.75)
# LSTM (offset by seq_len)
if len(lstm_preds) >= n_plot:
    ax.plot(x, lstm_preds[:n_plot], color='#43a047', linewidth=1, linestyle='-',
            label=f'LSTM (RMSE={rmse(yte_s,lstm_preds):.2f})', alpha=0.75)

ax.axhline(85, color='grey', linestyle='--', linewidth=1.2, label='SLA threshold (85%)')
ax.set_xlabel('Time Window (10-min intervals)')
ax.set_ylabel('CPU Utilisation (%)')
ax.set_title('Predicted vs Actual CPU% — All Models (Alibaba Test Set, 30-min horizon)',
             fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig('data/figures/predictions_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Multi-Horizon Evaluation
mh = results.get('multi_horizon', {})
if mh:
    print('=== Multi-Horizon Evaluation (XGBoost) ===')
    for label, v in mh.items():
        print(f'  {label}: RMSE={v["rmse"]:.4f}  MAE={v["mae"]:.4f}  MAPE={v["mape"]:.2f}%')

    horizons = list(mh.keys())
    rmse_vals = [mh[h]['rmse'] for h in horizons]
    colors_bar = ['#43a047', '#e53935', '#ff8f00']

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(horizons, rmse_vals, color=colors_bar, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, rmse_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.2f}', ha='center', va='bottom', fontweight='bold')
    ax.set_xlabel('Forecast Horizon')
    ax.set_ylabel('RMSE (%)')
    ax.set_title('XGBoost RMSE by Forecast Horizon', fontweight='bold')
    plt.tight_layout()
    plt.savefig('data/figures/multi_horizon_rmse.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Statistical Significance (Wilcoxon Signed-Rank Test)

In [ ]:
# Compute per-sample absolute errors for Wilcoxon test
n = min(len(y_test), len(rf_preds), len(xgb_preds))
xgb_err = np.abs(y_test[:n] - xgb_preds[:n])
rf_err  = np.abs(y_test[:n] - rf_preds[:n])
# LSTM uses same y_test subset as yte_s
n_lstm = min(n, len(yte_s), len(lstm_preds))
lstm_err = np.abs(yte_s[:n_lstm] - lstm_preds[:n_lstm])

pairs = [
    ('XGBoost vs RF',   xgb_err[:n], rf_err[:n]),
    ('XGBoost vs LSTM', xgb_err[:n_lstm], lstm_err[:n_lstm]),
    ('RF vs LSTM',      rf_err[:n_lstm],  lstm_err[:n_lstm]),
]
print('=== Wilcoxon Signed-Rank Test (α = 0.05) ===')
print(f'{"Comparison":<25} {"W-stat":>15} {"p-value":>12} {"Significant":>12}')
print('-' * 65)
for name, a, b in pairs:
    W, p = wilcoxon(a, b)
    sig = 'YES ✓' if p < 0.05 else 'no'
    print(f'{name:<25} {W:>15.1f} {p:>12.6f} {sig:>12}')

## 8. Cross-Cloud Generalisation & Transfer Learning

In [ ]:
cc = results.get('cross_cloud', {})
tl = results.get('transfer_learning', {})

print('=== Cross-Cloud Generalisation (Zero-Shot) ===')
print(f'{"Provider":<12} {"XGBoost RMSE":>14} {"RF RMSE":>10} {"LSTM RMSE":>10}')
print('-' * 50)
for prov, vals in cc.items():
    print(f'{prov.capitalize():<12} {vals.get("xgboost_rmse","—"):>14.4f} '
          f'{vals.get("rf_rmse","—"):>10.4f} {vals.get("lstm_rmse","—"):>10.4f}')

print()
print('=== Transfer Learning — 10% Fine-Tuning ===')
print(f'{"Provider":<12} {"Zero-Shot":>12} {"Fine-Tuned":>12} {"Improvement":>12}')
print('-' * 52)
for prov, vals in tl.items():
    print(f'{prov.capitalize():<12} {vals.get("rmse_zero_shot","—"):>12.4f} '
          f'{vals.get("rmse_after_finetune","—"):>12.4f} '
          f'{vals.get("improvement","—"):>12.4f}')

In [ ]:
# Cross-cloud heatmap
if Path('data/figures/cross_cloud_heatmap.png').exists():
    img = plt.imread('data/figures/cross_cloud_heatmap.png')
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('Cross-Cloud Generalisation Heatmap (RMSE)', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Run src/explainability/shap_analysis.py to generate heatmap')

## 9. SHAP Explainability

SHAP (SHapley Additive exPlanations) values quantify each feature's contribution to each prediction.

In [ ]:
# SHAP global feature importance — XGBoost
if Path('data/results/shap_xgboost.pkl').exists():
    shap_data = joblib.load('data/results/shap_xgboost.pkl')
    sv   = np.abs(shap_data['shap_values']).mean(axis=0)
    fns  = shap_data['feature_names']
    top_idx = np.argsort(sv)[-15:]

    fig, ax = plt.subplots(figsize=(10, 6))
    y_pos = range(len(top_idx))
    ax.barh(y_pos, sv[top_idx], color='#1e88e5', alpha=0.85, edgecolor='white')
    ax.set_yticks(y_pos)
    ax.set_yticklabels([fns[i].replace('_',' ') for i in top_idx], fontsize=9)
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title('XGBoost SHAP Global Feature Importance (Top 15 — Alibaba)',
                 fontweight='bold')
    plt.tight_layout()
    plt.savefig('data/figures/shap_xgb_global.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Run src/explainability/shap_analysis.py to generate SHAP values')

In [ ]:
# Display saved SHAP beeswarm and waterfall plots
for fname, title in [
    ('data/figures/shap_xgb_beeswarm.png', 'SHAP Beeswarm — XGBoost (Feature Value vs Impact)'),
    ('data/figures/shap_xgb_waterfall.png', 'SHAP Waterfall — Single High-CPU Prediction Explained'),
    ('data/figures/shap_comparison.png', 'SHAP Feature Importance Comparison — All 3 Models'),
]:
    if Path(fname).exists():
        img = plt.imread(fname)
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.imshow(img); ax.axis('off'); ax.set_title(title, fontweight='bold')
        plt.tight_layout(); plt.show()
    else:
        print(f'Missing: {fname} — run shap_analysis.py')

## 10. Cost Saving Analysis

Quantified daily € saving of proactive ML scaling vs reactive threshold scaling (AWS m5.xlarge, EU-West-1 Dublin, €0.192/hr).

In [ ]:
PRICE_PER_HOUR = 0.192   # AWS m5.xlarge EU-West-1

def cost_saving(predicted_cpu, actual_cpu, forecast_cpu,
                target_util=75, current_nodes=3):
    p_nodes = max(current_nodes,
                  int(np.ceil(forecast_cpu / target_util * current_nodes)))
    r_nodes = max(current_nodes,
                  int(np.ceil(actual_cpu   / target_util * current_nodes * 1.4)))
    saving_per_hour = max(0, r_nodes - p_nodes) * PRICE_PER_HOUR
    saving_per_day  = saving_per_hour * 24
    return round(saving_per_day, 4), p_nodes, r_nodes

ca = results.get('cost_analysis', {})
print('=== Cost Saving Analysis ===')
for model, vals in ca.items():
    print(f'  {model.capitalize():15s}: €{vals["mean_daily_saving_eur"]:.2f}/day  '
          f'CI=[€{vals["ci_lower"]:.2f}, €{vals["ci_upper"]:.2f}]  '
          f'{vals["pct_windows_proactive_cheaper"]:.1f}% windows cheaper')

# Bar chart
if Path('data/figures/cost_analysis.png').exists():
    img = plt.imread('data/figures/cost_analysis.png')
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.imshow(img); ax.axis('off')
    ax.set_title('Cost Saving Analysis — Proactive vs Reactive Scaling',
                 fontweight='bold')
    plt.tight_layout(); plt.show()

## 11. Error Analysis

Residual distribution, errors by CPU regime, and calibration plot.

In [ ]:
# Residual analysis for XGBoost
residuals = y_test - xgb_preds
bias = float(np.mean(residuals))
mae_val = float(np.mean(np.abs(residuals)))
rmse_val = float(np.sqrt(np.mean(residuals**2)))

print(f'XGBoost Error Analysis (Alibaba Test Set):')
print(f'  Mean Bias: {bias:+.4f}%  ({"over-predicts" if bias > 0 else "under-predicts"} — '
      f'{"safe for SLA" if bias > 0 else "risky for SLA"})')
print(f'  MAE:  {mae_val:.4f}%')
print(f'  RMSE: {rmse_val:.4f}%')
print(f'  Over-predictions:  {(residuals < 0).mean()*100:.1f}%')
print(f'  Under-predictions: {(residuals > 0).mean()*100:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residual histogram
axes[0].hist(residuals, bins=60, color='#1e88e5', alpha=0.8, edgecolor='white')
axes[0].axvline(0,    color='black', linestyle='--', linewidth=1.5, label='Zero error')
axes[0].axvline(bias, color='red',   linestyle='--', linewidth=1.5,
                label=f'Mean bias {bias:+.2f}%')
axes[0].set_xlabel('Residual (Actual − Predicted) %')
axes[0].set_ylabel('Count')
axes[0].set_title('Residual Distribution', fontweight='bold')
axes[0].legend()

# Predicted vs Actual scatter
axes[1].scatter(y_test, xgb_preds, alpha=0.15, s=2, color='#e53935')
lo = min(y_test.min(), xgb_preds.min())
hi = max(y_test.max(), xgb_preds.max())
axes[1].plot([lo,hi],[lo,hi], 'k--', linewidth=1.5, label='Perfect prediction')
axes[1].axvline(85, color='red', linestyle=':', linewidth=1.2, label='SLA 85%')
axes[1].set_xlabel('Actual CPU %')
axes[1].set_ylabel('Predicted CPU %')
axes[1].set_title('Calibration Plot — Predicted vs Actual', fontweight='bold')
axes[1].legend()

plt.suptitle('XGBoost Error Analysis', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('data/figures/error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Error by CPU regime
bins   = [0, 40, 60, 75, 85, 100]
labels = ['Low\n(0–40%)', 'Moderate\n(40–60%)', 'Elevated\n(60–75%)',
          'High\n(75–85%)', 'Critical\n(>85%)']
regime = pd.cut(y_test, bins=bins, labels=labels, include_lowest=True)
regime_df = pd.DataFrame({'regime': regime, 'abs_error': np.abs(residuals),
                           'residual': residuals})
reg_summary = regime_df.groupby('regime', observed=True).agg(
    Count  = ('abs_error', 'count'),
    MAE    = ('abs_error', 'mean'),
    RMSE   = ('abs_error', lambda x: np.sqrt((x**2).mean())),
    Bias   = ('residual', 'mean')
).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = ['#43a047','#1e88e5','#ff8f00','#f4511e','#e53935']
bars = ax.bar(reg_summary['regime'], reg_summary['MAE'],
              color=bar_colors, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, reg_summary['MAE']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=9)
ax.set_xlabel('CPU Utilisation Regime')
ax.set_ylabel('Mean Absolute Error (%)')
ax.set_title('XGBoost MAE by CPU Regime — SLA Risk Zones', fontweight='bold')
plt.tight_layout()
plt.savefig('data/figures/error_by_regime.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nError breakdown by CPU regime:')
print(reg_summary.round(3).to_string(index=False))

## 12. Learning Curves

RMSE vs training set size (10/25/50/75/100%) for all three models.

In [ ]:
lc = joblib.load('data/results/learning_curves.pkl')

fractions = [f * 100 for f in lc['fraction']]

fig, ax = plt.subplots(figsize=(10, 5))
model_colors = {'xgb': '#e53935', 'rf': '#1e88e5', 'lstm': '#43a047'}
model_labels = {'xgb': 'XGBoost', 'rf': 'Random Forest', 'lstm': 'LSTM'}

for key in ['xgb', 'rf', 'lstm']:
    if key in lc:
        ax.plot(fractions, lc[key], 'o-',
                color=model_colors[key], linewidth=2, markersize=6,
                label=model_labels[key])

ax.set_xlabel('Training Set Size (%)')
ax.set_ylabel('RMSE on Test Set (%)')
ax.set_title('Learning Curves — RMSE vs Training Set Size (Alibaba)',
             fontweight='bold')
ax.legend(fontsize=10)
ax.set_xticks(fractions)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('data/figures/learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print('Learning curve values:')
for key in ['xgb', 'rf', 'lstm']:
    if key in lc:
        print(f'  {model_labels[key]:15s}: {[round(v,2) for v in lc[key]]}')

In [ ]:
print('\n' + '='*60)
print('H9MLAI — Pipeline Complete')
print('='*60)
print('Sabhyata Kumari | X24283142 | NCI MSc AI 2025/2026')
print('\nKey Results:')
for name, r in results['model_performance'].items():
    print(f"  {name.replace('_',' ').title():15s}: RMSE={r['rmse']:.2f}%  "
          f"Latency={r['latency_ms']:.1f}ms")
tl = results.get('transfer_learning', {})
if tl:
    print('\nTransfer Learning (XGBoost fine-tuned on 10% target data):')
    for p, v in tl.items():
        print(f"  {p.capitalize()}: {v['rmse_zero_shot']:.2f} → {v['rmse_after_finetune']:.2f} RMSE")
ca = results.get('cost_analysis', {})
if ca:
    xgb_ca = ca.get('xgboost', {})
    print(f"\nCost Saving (XGBoost): €{xgb_ca.get('mean_daily_saving_eur', 0):.2f}/day "
          f"({xgb_ca.get('pct_windows_proactive_cheaper', 0):.1f}% windows cheaper)")